# Customer Churn Prediction & Retention System
Experimentation notebook. Reusable final code lives in `src/`; this notebook shows the reasoning and results.

**Rules followed:** no test data during training/selection, preprocessing inside pipelines (no leakage), every finding comes from *your* run of the data (nothing is pre-filled).

> Colab: upload the CSV and the `src/` folder, or clone your repo, and set `DATA_PATH` accordingly.

In [ ]:
import sys, warnings
sys.path.insert(0, "..")          # so `import src...` works from /notebooks
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

from src.preprocessing import *
DATA_PATH = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
SEED = RANDOM_STATE

## Milestone 1 — Load and understand the data
`head` = do the values look right? `shape` = how much data? `dtypes/info` = are types right, any nulls? `describe` = ranges and spread of numeric columns.

In [ ]:
df_raw = load_data(DATA_PATH)
display(df_raw.head())
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
print(df_raw.dtypes)
df_raw.info()
display(df_raw.describe())

## Milestone 2 — Data quality checks and cleaning

In [ ]:
print("Missing values:\n", df_raw.isnull().sum()[lambda s: s > 0], "\n")
print("Duplicate rows:", df_raw.duplicated().sum())
print("Duplicate customerIDs:", df_raw["customerID"].duplicated().sum(), "\n")
for col in df_raw.columns:
    print(f"{col:20s} {df_raw[col].nunique()}")

### Target distribution
Look at the actual numbers before calling it balanced or imbalanced.

In [ ]:
print(df_raw["Churn"].value_counts())
print(df_raw["Churn"].value_counts(normalize=True).round(3))

### Why is `TotalCharges` text?
Investigate *before* filling anything. Hypothesis: blank strings for customers with `tenure == 0` (not yet billed).

In [ ]:
blank = df_raw["TotalCharges"].str.strip() == ""
print("Blank TotalCharges rows:", blank.sum())
display(df_raw.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]])
print("tenure values among blank rows:", df_raw.loc[blank, "tenure"].unique())

**Verify:** if every blank row has `tenure == 0`, the missingness has a clear cause (new customers, nothing billed yet), so `0` is a defensible value. `FeatureEngineer` applies exactly this rule; any *other* missing values go to the median imputer inside the pipeline.

Cleaning (`clean_data`): drop `customerID` (identifier), convert `TotalCharges` to numeric, map target `Yes/No -> 1/0`. These steps learn nothing from the data, so doing them before the split is safe.

In [ ]:
df = clean_data(df_raw)
print(df.shape, "| TotalCharges NaNs after conversion:", df["TotalCharges"].isna().sum())
print(df["Churn"].mean().round(3), "= overall churn rate")

## Milestone 3 — Exploratory Data Analysis
Charts show **association, not causation**. For each one, write down your own interpretation from the output.

In [ ]:
# EDA uses the TRAINING split only, so test-set patterns can't influence our decisions
X_tr_eda, _, y_tr_eda, _ = make_split(*split_xy(df))
eda = X_tr_eda.assign(Churn=y_tr_eda)
eda["Churn_label"] = eda["Churn"].map({1: "Churned", 0: "Stayed"})

fig, ax = plt.subplots(figsize=(4, 3.5))
eda["Churn_label"].value_counts().plot.bar(ax=ax, color=["tab:blue", "tab:red"])
ax.set_title("Churn distribution"); ax.set_ylabel("Customers"); plt.xticks(rotation=0); plt.show()

In [ ]:
def churn_rate_by(col):
    t = eda.groupby(col)["Churn"].agg(["mean", "count"]).rename(columns={"mean": "churn_rate"})
    return t.round(3)

for col in ["Contract", "InternetService", "PaymentMethod"]:
    print(churn_rate_by(col), "\n")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    sns.countplot(data=eda, x=col, hue="Churn_label", ax=ax)
    ax.set_title(f"Churn by {col}"); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
sns.histplot(data=eda, x="tenure", hue="Churn_label", bins=30, ax=axes[0], element="step")
sns.boxplot(data=eda, x="Churn_label", y="tenure", ax=axes[1])
sns.boxplot(data=eda, x="Churn_label", y="MonthlyCharges", ax=axes[2])
sns.boxplot(data=eda, x="Churn_label", y="TotalCharges", ax=axes[3])
for ax, t in zip(axes, ["Tenure distribution", "Tenure", "Monthly charges", "Total charges"]):
    ax.set_title(t)
plt.tight_layout(); plt.show()
print(eda.groupby("Churn_label")[["tenure", "MonthlyCharges", "TotalCharges"]].median().round(1))

In [ ]:
plt.figure(figsize=(5, 4))
sns.heatmap(df[["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn"]].corr(),
            annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation (numeric features)"); plt.show()

**Note:** `TotalCharges` is largely `tenure x MonthlyCharges`, so they are strongly correlated. That's fine for tree models and tolerable for regularised Logistic Regression, but it makes individual coefficients harder to interpret.

## Milestone 4 — Feature engineering
Engineered inside `FeatureEngineer` (see `src/preprocessing.py`): `tenure_group`, `avg_monthly_revenue`, `service_count`, `has_protection_support`. None uses the target. `Contract` stays one-hot encoded (no arbitrary numbers); `SeniorCitizen` is treated as a categorical yes/no.

In [ ]:
fe = FeatureEngineer().transform(df.drop(columns="Churn"))
display(fe[["tenure", "tenure_group", "TotalCharges", "avg_monthly_revenue", "service_count", "has_protection_support"]].head(8))
print("tenure==0 rows ->", fe.loc[df["tenure"] == 0, ["TotalCharges", "avg_monthly_revenue"]].drop_duplicates().to_dict("records")[:1])
tmp = fe.assign(Churn=df["Churn"].values)
print(tmp.groupby("tenure_group")["Churn"].mean().round(3))
print(tmp.groupby("service_count")["Churn"].agg(["mean", "count"]).round(3))

## Milestone 5 — Split first, then build the pipeline
`stratify=y` keeps the churn proportion the same in train and test, so evaluation isn't distorted by an unlucky split.
Training data -> model learns. Test data -> final evaluation only.
The pipeline (`FeatureEngineer -> ColumnTransformer(impute, scale, one-hot) -> model`) is fitted **only on training data** inside `fit()`, which prevents leakage.

In [ ]:
from sklearn.model_selection import train_test_split
X, y = split_xy(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(X_train.shape, X_test.shape, "| churn rate train/test:", y_train.mean().round(3), y_test.mean().round(3))
print(build_preprocessor())

## Milestone 6/7 — Baselines and comparison
Candidates (`src/train.py`): Logistic Regression (interpretable baseline), Decision Tree, Random Forest, XGBoost (if installed).
`class_weight="balanced"` is used so the minority (churn) class counts more; this typically raises recall and lowers precision.
Models are compared with **5-fold CV on the training set**, so the test set isn't used for selection.

In [ ]:
from sklearn.model_selection import cross_validate
from src.train import get_candidates, CV, SCORING, METRIC_LABELS
candidates = get_candidates(y_train)
rows = []
for name, (model, _) in candidates.items():
    s = cross_validate(build_pipeline(model), X_train, y_train, cv=CV, scoring=SCORING)
    rows.append({"Model": name, **{lab: s[f"test_{k}"].mean() for k, lab in METRIC_LABELS.items()}})
cv_table = pd.DataFrame(rows).set_index("Model").round(4)
display(cv_table)

### Which metric matters?
- **False negative** = a churner we missed (lost revenue, no chance to intervene).
- **False positive** = a loyal customer we flagged (cost of an unnecessary offer/contact).

If missing churners is costlier than contacting extra customers, prioritise **recall** (while watching precision/F1). ROC-AUC measures ranking quality across all thresholds. The true costs are a business input; this project doesn't know them, so decide using the threshold table below rather than assuming.

## Milestone 8 — Hyperparameter tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
best_name = cv_table["ROC-AUC"].idxmax()
model, space = candidates[best_name]
print("Tuning:", best_name)
search = RandomizedSearchCV(build_pipeline(model), space, n_iter=25, scoring="roc_auc",
                            cv=CV, random_state=SEED, n_jobs=-1)
search.fit(X_train, y_train)
print(search.best_params_); print("CV ROC-AUC:", round(search.best_score_, 4))
final_pipeline = search.best_estimator_

*Parameter* = learned from data (e.g., coefficients). *Hyperparameter* = set before training (e.g., `max_depth`). *Cross-validation* = repeatedly train on part of the training data and validate on the held-out fold. *Overfitting* = great on training data, worse on new data; we limit it with CV-based selection, depth/leaf constraints and a held-out test set.

## Milestone 9 — Final evaluation (test set, used once)

In [ ]:
from src.evaluate import *
rows, probs = [], {}
for name, (m, _) in candidates.items():
    f = build_pipeline(m).fit(X_train, y_train)
    rows.append(evaluate_pipeline(name, f, X_test, y_test)); probs[name] = f.predict_proba(X_test)[:, 1]
rows.append(evaluate_pipeline(f"{best_name} (tuned)", final_pipeline, X_test, y_test))
probs[f"{best_name} (tuned)"] = final_pipeline.predict_proba(X_test)[:, 1]
display(pd.DataFrame(rows).set_index("Model").round(4))

y_prob = final_pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)
print(report(y_test, y_pred))
plot_confusion_matrix(y_test, y_pred); plt.show()
plot_roc_curves(probs, y_test); plt.show()
display(threshold_table(y_test, y_prob).round(3))

**Confusion matrix:** TN = stayed & predicted stay; FP = stayed but flagged; FN = churned but missed; TP = churned & caught.
Threshold 0.5 is the default, not a law; the table above shows how precision/recall move. If the business prefers higher recall, lower the threshold and accept more false positives.

## Milestone 10 — SHAP explainability
SHAP splits a prediction into per-feature contributions (from game theory). It explains **model behaviour, not causation**. Requires `pip install shap`.

In [ ]:
import shap
from src.explain import *
bg_raw = X_train.sample(100, random_state=SEED)
bg, names = transform_features(final_pipeline, bg_raw)
explainer = build_explainer(final_pipeline, bg)

sample = X_test.sample(300, random_state=SEED)
mat, names = transform_features(final_pipeline, sample)
vals = shap_values(explainer, mat)
shap.summary_plot(vals, mat, feature_names=names, max_display=15)
display(global_importance(final_pipeline, explainer, sample).head(10))

In [ ]:
one = X_test.iloc[[0]]
p = final_pipeline.predict_proba(one)[0, 1]
up, down = explain_customer(final_pipeline, explainer, one)
print(f"Churn probability: {p:.1%}")
print("Increasing risk:", [(k, round(v, 3)) for k, v in up])
print("Reducing risk:  ", [(k, round(v, 3)) for k, v in down])

## Milestone 11 — Risk segmentation and recommendations
Thresholds (0.30 / 0.70) are project-defined and configurable. The rules are business heuristics layered on the model, **not** causal conclusions.

In [ ]:
from src.recommendations import risk_level, recommend
cust = one.iloc[0].to_dict()
print(f"Churn Probability: {p:.2f} | Risk Level: {risk_level(p)}")
for a in recommend(cust, p): print("-", a)
seg = pd.Series(y_prob).map(risk_level).value_counts()
print(seg)
print(pd.DataFrame({"risk": pd.Series(y_prob).map(risk_level).values, "churned": y_test.values}).groupby("risk")["churned"].agg(["mean", "count"]))

## Milestone 12 — Save the full pipeline
The saved object includes feature engineering + preprocessing, so the app passes raw input. Prefer `python -m src.train` for the canonical saved model.

In [ ]:
import joblib, os
os.makedirs("../models", exist_ok=True)
joblib.dump(final_pipeline, "../models/churn_model.pkl")
reloaded = joblib.load("../models/churn_model.pkl")
print(reloaded.predict_proba(X_test.head(3))[:, 1])